# Mixed-Effects Statistical Modeling — WER across Language × DataRegime
Data: `mixed_effects_data_matrix.csv` (24 utterances, 6 speakers, 24 items).
Languages: **HEB_SCO** (Hebridean Scottish English / Gaelic contact) and **AZB** (South Azerbaijani).
Key issue: Language is *confounded* with DataRegime, so the full fixed model is singular; we fit a fallback with
speaker random intercepts and run diagnostics.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import statsmodels.formula.api as smf
mx = pd.read_csv('/mnt/data/mixed_effects_data_matrix.csv')
print(mx.shape, mx.Language.value_counts().to_dict())
pd.crosstab(mx.Language, mx.DataRegime)

In [ ]:
# Cross-tab shows the rank deficiency: AZB only in 3 regimes, HEB_SCO only in 2.
X = pd.get_dummies(mx[['Language','DataRegime']], drop_first=True)
print('rank:', np.linalg.matrix_rank(X.values), 'cols:', X.shape[1])

### Full fixed-effects attempt (expected singular)

In [ ]:
try:
    full = smf.mixedlm("wer ~ C(Language) + C(DataRegime_Group)", mx, groups=mx['Speaker']).fit(method='powell')
    print(full.summary())
except Exception as e:
    print('NOT ESTIMABLE:', type(e).__name__, e)

In [ ]:
# Fallback: regime-only fixed effects + speaker random intercept (ML, Powell)
m1 = smf.mixedlm("wer ~ C(DataRegime_Group)", mx, groups=mx['Speaker']).fit(method='powell', reml=False)
print(m1.summary())
print('Group var (speaker):', m1.cov_re.iloc[0,0], ' Residual scale:', m1.scale)

In [ ]:
# OLS contrast for descriptive language gap within the shared regime structure
ols = smf.ols("wer ~ C(DataRegime_Group)", mx).fit()
print(ols.summary().tables[1])

### Diagnostics: residuals, Q–Q, per-speaker caterpillar plot

In [ ]:
import scipy.stats as st
mx['fit'] = m1.fittedvalues; mx['resid'] = m1.resid
fig, ax = plt.subplots(1,3, figsize=(15,4))
ax[0].scatter(mx.fit, mx.resid); ax[0].axhline(0, c='r'); ax[0].set(title='Residuals vs fitted', xlabel='fitted WER')
st.probplot(mx.resid, plot=ax[1]); ax[1].set_title('Q-Q plot of residuals')
bl = m1.random_effects; names=[f'g{k[0] if isinstance(k,tuple) else k}' for k in bl.keys()]
sp = mx.groupby('Speaker')['resid'].mean()
ax[2].barh(range(len(sp)), sp.values); ax[2].set_yticks(range(len(sp))); ax[2].set_yticklabels(sp.index, fontsize=7)
ax[2].axvline(0,c='r'); ax[2].set_title('Per-speaker mean residual (caterpillar)')
plt.tight_layout(); plt.savefig('/mnt/data/notebooks/figure_diagnostics.png', dpi=150); plt.show()

### Interpretation
- Speaker random-intercept variance sits near the **boundary of zero** → with only 6 speakers the random effect is not
  reliably estimable; results are descriptive.
- **Language × DataRegime interaction is structurally inestimable** in this corpus (perfect confounding). Any
  cross-linguistic WER claim must come from a design that crosses Language with Regime (more AZB archival data, or
  HEB_SCO spontaneous data).